### Prepare environment

In [ ]:
%run ../../environment/prepare_environment_llm

# Notebook 9.3 - RAG Chatbot

Create or reuse a Databricks AI Search index, retrieve workshop context, and answer questions with citations.

RAG changes the LLM workflow from "answer from model memory" to "retrieve relevant governed context, then answer from that context". This notebook shows the retrieval step, prompt assembly, and grounded generation step separately.

## 1. Configure Retrieval and Model Endpoints

The chatbot needs three moving parts: a chunks table, an AI Search index over that table, and a Foundation Model endpoint for generation. Widgets make those names explicit so the same notebook can be reused in another catalog or schema.

In [ ]:
import math
import re
import time
from typing import Any

from databricks.ai_search.client import VectorSearchClient
from mlflow.deployments import get_deploy_client

dbutils.widgets.text("catalog_name", "ai_ml_in_practice")
dbutils.widgets.text("schema_name", "genai_workshop")
dbutils.widgets.text("vector_search_endpoint_name", "genai-workshop-search")
dbutils.widgets.text("embedding_model_endpoint_name", "databricks-gte-large-en")
dbutils.widgets.text("llm_endpoint_name", "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.dropdown("create_or_update_index", "true", ["true", "false"])
dbutils.widgets.text("top_k", "4")

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
vector_search_endpoint_name = dbutils.widgets.get("vector_search_endpoint_name")
embedding_model_endpoint_name = dbutils.widgets.get("embedding_model_endpoint_name")
llm_endpoint_name = dbutils.widgets.get("llm_endpoint_name")
create_or_update_index = dbutils.widgets.get("create_or_update_index") == "true"
top_k = int(dbutils.widgets.get("top_k"))

chunks_table = f"{catalog_name}.{schema_name}.document_chunks"
index_name = f"{catalog_name}.{schema_name}.document_chunks_rag_index"

vector_search_client = VectorSearchClient(disable_notice=True)
deploy_client = get_deploy_client("databricks")

print(
    {
        "chunks_table": chunks_table,
        "index_name": index_name,
        "vector_search_endpoint_name": vector_search_endpoint_name,
        "embedding_model_endpoint_name": embedding_model_endpoint_name,
        "llm_endpoint_name": llm_endpoint_name,
    }
)

## 2. Create or Reuse the AI Search Index

The Delta Sync index turns text chunks into searchable vectors using a Databricks embedding endpoint. The source table remains the governed system of record; the index is the serving structure used for low-latency semantic retrieval.

In [ ]:
def object_to_dict(value: Any) -> dict[str, Any]:
    """Convert Databricks SDK objects into plain dictionaries.

    SDK methods can return dictionaries, generated classes, or objects with
    helper methods such as `as_dict`. Status checks are easier to explain and
    print when each response is normalized to the same Python shape.
    """
    if isinstance(value, dict):
        return value
    if hasattr(value, "as_dict"):
        return value.as_dict()
    if hasattr(value, "__dict__"):
        return dict(value.__dict__)
    return {}


def ensure_vector_search_endpoint(endpoint_name: str) -> None:
    """Ensure the serving endpoint for vector search exists.

    The endpoint is the compute resource behind the vector index. The function
    first tries to reuse an existing endpoint, then creates one only when needed,
    and finally polls until the endpoint can accept indexes.
    """
    try:
        vector_search_client.get_endpoint(name=endpoint_name)
        print(f"Vector Search endpoint exists: {endpoint_name}")
        return
    except Exception as exc:
        print(f"Creating Vector Search endpoint {endpoint_name}: {exc}")

    vector_search_client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
    for _ in range(60):
        endpoint = object_to_dict(vector_search_client.get_endpoint(name=endpoint_name))
        state = str(
            endpoint.get("endpoint_status", {}).get("state", endpoint.get("state", ""))
        ).upper()
        print({"endpoint": endpoint_name, "state": state})
        if state in {"ONLINE_NO_PENDING_UPDATE", "PROVISIONING_INDEXES", "READY"}:
            return
        time.sleep(10)


def ensure_delta_sync_index() -> Any:
    """Create or refresh the Delta Sync index used by retrieval.

    The Delta table remains the source of truth for chunks and metadata. The
    index is a serving structure: it embeds `retrieval_text`, stores vectors, and
    makes semantic similarity search fast enough for chatbot-style interaction.
    """
    try:
        index = vector_search_client.get_index(
            endpoint_name=vector_search_endpoint_name,
            index_name=index_name,
        )
        print(f"Vector Search index exists: {index_name}")
        if hasattr(index, "sync"):
            index.sync()
        return index
    except Exception as exc:
        print(f"Creating Vector Search index {index_name}: {exc}")

    return vector_search_client.create_delta_sync_index(
        endpoint_name=vector_search_endpoint_name,
        index_name=index_name,
        source_table_name=chunks_table,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="retrieval_text",
        embedding_model_endpoint_name=embedding_model_endpoint_name,
    )


def wait_for_index_ready(index: Any, timeout_seconds: int = 900) -> Any:
    """Wait until the index reports a searchable state.

    Index creation and synchronization are asynchronous operations. Polling keeps
    the next cells honest: retrieval should run after the index has had a chance
    to embed and publish the chunks.
    """
    start = time.time()
    while time.time() - start < timeout_seconds:
        description = (
            object_to_dict(index.describe()) if hasattr(index, "describe") else {}
        )
        status = object_to_dict(description.get("status", {}))
        detailed_state = str(
            status.get("detailed_state", status.get("state", ""))
        ).upper()
        print({"index": index_name, "state": detailed_state})
        if detailed_state in {"ONLINE_NO_PENDING_UPDATE", "READY"}:
            return index
        time.sleep(15)
    return index


if create_or_update_index:
    ensure_vector_search_endpoint(vector_search_endpoint_name)
    vector_index = ensure_delta_sync_index()
    vector_index = wait_for_index_ready(vector_index)
else:
    vector_index = vector_search_client.get_index(
        endpoint_name=vector_search_endpoint_name,
        index_name=index_name,
    )

## 3. Define Retrieval Functions

The retriever combines three simple ideas:

1. AI Search returns a larger semantic candidate set from `retrieval_text`.
2. Delta lexical retrieval scans the governed chunks table using important query terms.
3. A lightweight reranker boosts candidates whose metadata and text match important query terms.

A query such as `Where does this workshop showcase classic ML algorithms?` should match module metadata (`ML Algorithms - The Classical Approach`) even if the exact phrase is not repeated in every notebook chunk.

In [ ]:
SEARCH_STOPWORDS = {
    "a",
    "an",
    "and",
    "are",
    "as",
    "at",
    "be",
    "by",
    "does",
    "for",
    "from",
    "how",
    "in",
    "is",
    "it",
    "of",
    "on",
    "or",
    "the",
    "this",
    "to",
    "what",
    "where",
    "which",
    "who",
    "why",
    "with",
    "workshop",
    "show",
    "showcase",
    "showcases",
}


def extract_query_terms(query: str) -> set[str]:
    """Extract meaningful lowercase terms used by lexical reranking.

    Very common words such as `where` and `workshop` appear in many questions
    and do not help identify the right chunk. Removing them makes terms like
    `classic`, `ml`, and `algorithms` dominate the reranking step.
    """
    return {
        term
        for term in re.findall(r"[a-zA-Z0-9]+", query.lower())
        if term not in SEARCH_STOPWORDS and len(term) > 1
    }


def searchable_text(row: dict[str, Any]) -> str:
    """Join content and metadata fields used for lexical scoring.

    The vector index embeds `retrieval_text`, but reranking should also consider
    returned metadata. This joined text lets the scorer reward module titles,
    source paths, section names, and the actual chunk passage at the same time.
    """
    return "\n".join(
        str(row.get(field, ""))
        for field in [
            "module_id",
            "module_title",
            "source_path",
            "section_title",
            "retrieval_text",
            "chunk_text",
        ]
    ).lower()


def lexical_score(query: str, row: dict[str, Any]) -> float:
    """Score how directly a candidate matches the user's wording.

    Vector search is good at semantic similarity, but short workshop questions
    often contain strong lexical clues such as a module number or phrase. This
    score boosts candidates whose metadata and retrieval text contain those
    clues, then the final retriever combines this with vector candidates.
    """
    terms = extract_query_terms(query)
    text = searchable_text(row)
    if not terms:
        return 0.0
    score = sum(
        2.0 if term in str(row.get("module_title", "")).lower() else 0.0
        for term in terms
    )
    score += sum(1.0 for term in terms if term in text)
    if "classic" in terms and "classical" in text:
        score += 2.0
    module_match = re.search(r"module\s+(\d+)", query.lower())
    if module_match and str(row.get("module_id")) == module_match.group(1):
        score += 5.0
    return score


def rerank_chunks(
    query: str, chunks: list[dict[str, Any]], k: int
) -> list[dict[str, Any]]:
    """Deduplicate and rerank candidate chunks before prompt assembly.

    Vector search and Delta lexical retrieval can return overlapping chunks. We keep
    one copy per `chunk_id`, attach a local `_rerank_score` for inspection, and
    return the strongest candidates for the LLM context window.
    """
    by_id = {}
    for chunk in chunks:
        chunk_id = chunk.get("chunk_id")
        if chunk_id and chunk_id not in by_id:
            by_id[chunk_id] = dict(chunk)
    reranked = []
    for chunk in by_id.values():
        chunk["_rerank_score"] = lexical_score(query, chunk)
        reranked.append(chunk)
    return sorted(reranked, key=lambda row: row["_rerank_score"], reverse=True)[:k]


def parse_similarity_response(response: dict[str, Any]) -> list[dict[str, Any]]:
    """Convert AI Search tabular output into chunk dictionaries.

    The search API returns column names in a manifest and row values in a separate
    array. RAG code is clearer when each retrieved chunk is one dictionary with
    text and citation metadata in the same object.
    """
    manifest = response.get("manifest", {})
    columns = [column["name"] for column in manifest.get("columns", [])]
    rows = response.get("result", {}).get("data_array", [])
    return [dict(zip(columns, row, strict=False)) for row in rows]


def vector_search(query: str, k: int = top_k) -> list[dict[str, Any]]:
    """Retrieve chunks by semantic similarity.

    Unlike keyword search, vector search can match related wording: a question
    about `governed model lifecycle` can retrieve text that says `register and
    track models`. The returned columns are exactly the fields needed to build a
    grounded prompt and show citations.
    """
    response = vector_index.similarity_search(
        query_text=query,
        columns=[
            "chunk_id",
            "module_id",
            "module_title",
            "source_path",
            "section_title",
            "chunk_text",
            "retrieval_text",
        ],
        num_results=max(k * 5, 20),
    )
    return parse_similarity_response(response)


def delta_lexical_search(query: str, k: int = top_k) -> list[dict[str, Any]]:
    """Retrieve chunks from the Delta table using shared query terms.

    This is a transparent lexical retriever over the same governed chunk table
    used by AI Search. It only scores rows
    that were prepared in the knowledge-base notebook.
    """
    query_term_set = extract_query_terms(query)
    rows = (
        spark.table(chunks_table)
        .select(
            "chunk_id",
            "module_id",
            "module_title",
            "source_path",
            "section_title",
            "chunk_text",
            "retrieval_text",
        )
        .collect()
    )
    scored_rows = []
    for row in rows:
        row_dict = row.asDict()
        text_terms = set(re.findall(r"[a-zA-Z0-9]+", searchable_text(row_dict)))
        score = len(query_term_set & text_terms) / math.sqrt(max(len(text_terms), 1))
        score += lexical_score(query, row_dict)
        scored_rows.append((score, row_dict))
    return [
        row
        for score, row in sorted(scored_rows, key=lambda item: item[0], reverse=True)[
            : max(k * 5, 20)
        ]
        if score > 0
    ]


def retrieve(query: str, k: int = top_k) -> list[dict[str, Any]]:
    """Run the retrieval step used by the chatbot.

    RAG is a two-stage pattern: first retrieve relevant context, then ask an LLM
    to answer with that context. This function isolates stage one and provides a
    lexical retrieval path so the rest of the notebook can focus on prompt grounding.
    """
    try:
        candidates = vector_search(query, k)
    except Exception as exc:
        print(
            f"Vector Search unavailable; continuing with Delta lexical retrieval: {exc}"
        )
        candidates = []
    candidates.extend(delta_lexical_search(query, k))
    return rerank_chunks(query, candidates, k)

## 4. Inspect Retrieved Context

Before calling an LLM, inspect the retrieved chunks. If the chunks do not contain the information needed to answer the question, the generated answer will not be reliable.

In [ ]:
question = "Where does this workshop showcase classic ML algorithms?"
retrieved_chunks = retrieve(question, top_k)

display(spark.createDataFrame(retrieved_chunks))

## 5. Generate a Grounded Answer

The prompt contains a strict system instruction, retrieved context, and the user question. The model is asked to cite `source_path` values so the answer can be traced back to the workshop material.

In [ ]:
def build_context(chunks: list[dict[str, Any]]) -> str:
    """Format retrieved chunks into the prompt context block.

    The LLM receives plain text, so retrieval results must be serialized into a
    readable prompt section. Each chunk includes a number, source path, section
    title, and text so the model has enough information to cite its answer.
    """
    context_blocks = []
    for idx, chunk in enumerate(chunks, start=1):
        context_blocks.append(
            "\n".join(
                [
                    f"[{idx}] module={chunk.get('module_id')} title={chunk.get('module_title')} source_path={chunk.get('source_path')} section={chunk.get('section_title')}",
                    str(chunk.get("chunk_text", "")),
                ]
            )
        )
    return "\n\n".join(context_blocks)


def parse_llm_response(response: Any) -> str:
    """Extract answer text from common serving response shapes.

    Different serving endpoints can wrap generated text slightly differently.
    This helper keeps endpoint-specific parsing out of the RAG logic, so the
    main answer function still reads as retrieve, prompt, call, return.
    """
    if isinstance(response, dict):
        choices = response.get("choices", [])
        if choices:
            message = choices[0].get("message", {})
            return message.get("content", choices[0].get("text", str(response)))
        return str(response)
    return str(response)


def answer_with_rag(question: str, k: int = top_k) -> dict[str, Any]:
    """Answer a question with retrieval-grounded generation.

    The function makes the full RAG contract explicit: retrieve chunks, place
    them in the prompt, instruct the model to answer only from that context, and
    return both the generated answer and the sources that supported it.
    """
    chunks = retrieve(question, k)
    context = build_context(chunks)
    messages = [
        {
            "role": "system",
            "content": (
                "Answer only from the provided context. If the question context is different, "
                "say that the workshop material does not contain enough information. "
                "Cite source_path values used in the answer."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion:\n{question}"},
    ]
    response = deploy_client.predict(
        endpoint=llm_endpoint_name,
        inputs={"messages": messages, "temperature": 0.0, "max_tokens": 500},
    )
    return {
        "question": question,
        "answer": parse_llm_response(response),
        "sources": [chunk.get("source_path") for chunk in chunks],
        "retrieved_chunks": chunks,
    }

In [ ]:
question = "Where does this workshop showcase classic ML algorithms?"
rag_result = answer_with_rag(question)
print(rag_result["answer"])
rag_result["sources"]

## 6. Ask an Out-of-Scope Question

A useful RAG assistant should be able to say when the retrieved context is insufficient. This example checks whether the system instruction prevents the model from inventing an answer from outside the workshop material.

In [ ]:
out_of_scope_question = "What is the best restaurant near the Eiffel Tower today?"
out_of_scope_result = answer_with_rag(out_of_scope_question)
print(out_of_scope_result["answer"])
out_of_scope_result["sources"]